# Dashboard Contoso

## Importeren libraries en datasets

In [1]:
#Importeren van libraries
import matplotlib.pyplot as plt
import pandas as pd
import squarify
import seaborn as sns
import plotly.graph_objs as go
import plotly.io as pio
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px

In [2]:
#Importeren parquet files uit ETL.
sales_df = pd.read_parquet(r"PQ/Contoso_sales.parquet")
product_df = pd.read_parquet(r"PQ/Contoso_product.parquet")
calendar_df = pd.read_parquet(r"PQ/Contoso_calendar.parquet")
geography_df = pd.read_parquet(r"PQ/Contoso_geography.parquet")
stores_df = pd.read_parquet(r"PQ/Contoso_stores.parquet")
promotions_df = pd.read_parquet(r"PQ/Contoso_promotions.parquet")
productcat_df = pd.read_parquet(r"PQ/Contoso_productcat.parquet")
channel_df = pd.read_parquet(r"PQ/Contoso_channel.parquet")
GDP_df = pd.read_parquet(r"PQ/Contose_GDP.parquet")

## Sales Dashboard

### Measures

In [3]:
#In dit onderdeel worden de measures uit PowerBI nagemaakt. 
#De waarden voor revenue en cost worden eerst totaal berekend. Vervolgens worden Gross Margin en Gross margin in procenten op basis hiervan berekend.

revenue = sales_df["SalesAmountRet"].sum()
#Sales berekening
total_cost = sales_df["TotalCostRet"].sum()
#Cost berekening
gross_margin = revenue - total_cost
#Gross margin berekening
gross_margin_pct = (gross_margin / revenue)*100
#Gross margin in percentage

### KPI's

In [4]:
#Revenue KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=revenue,
        title={"text": "Revenue"},
        number={
            "prefix": "€",
            #Zet euro-teken voor het getal
            "valueformat": ",.0f"
            #Rond af op 0 decimalen.
        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen
fig.show()
#Cost KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=total_cost,
        title={"text": "Cost"},
        number={
            "prefix": "€",
            #Zet euro-teken voor het getal
            "valueformat": ",.0f"
            #Rond af op hele getallen en gebruikt komma's als scheidingstekens tussen duizendtallen.
        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen
fig.show()
#Gross margin KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=gross_margin,
        title={"text": "Gross Margin"},
        number={
            "prefix": "€",
            #Zet euro-teken voor het getal
            "valueformat": ",.0f"
            #Rond af op hele getallen en gebruikt komma's als scheidingstekens tussen duizendtallen.
        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen
fig.show()
#Gross margin percentage KPI
fig = go.Figure(
    #Maakt de figuur
    go.Indicator(
        #Maakt de KPI-visual
        mode="number",
        value=gross_margin_pct,
        title={"text": "Gross Margin percentage"},
        number={
            "suffix": "%",
            # %-teken na het percentage.
            "valueformat": ".01f"
            #Rond het percentage af op 1 decimaal.

        }
    )
)
fig.update_layout(
    height=200,
    width=300
)
#Formaat instellen

fig.show()


### Revenue over time

In [5]:
monthly_revenue = sales_df.resample("ME", on="Date").agg(Revenue=("SalesAmountRet","sum")).reset_index()
#Totale omzet aggregeren per maand en reset de index, zodat datum niet als index gebruikt blijft worden
monthly_revenue
monthly_revenue["Year"] = monthly_revenue["Date"].dt.year.astype(str)
#Maakt een kolom met de dataframe in jaar.
monthly_revenue["Month"] = monthly_revenue["Date"].dt.month
#Maakt een kolom met de maand van het jaar in nummers.

In [6]:
fig = px.line(
    monthly_revenue,
    x="Month",
    y="Revenue",
    color="Year",
    title="Revenue over time"
)
#Figuur maken met maanden op de x-as, omzet op de y-as en aparte lijnen voor ieder jaar.

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Revenue",
    height=400,
    width=1000
)
#Omschrijvingen van assen toevoegen.

fig.update_yaxes(
    tickprefix="€",
    tickformat=",.0f"
)
#Valuta maken van waarden op de y-as

fig.update_xaxes(
    tickmode="array",
    tickvals=list(range(1, 13)),
    ticktext=[
        "Jan", "Feb", "Mar", "Apr",
        "May", "Jun", "Jul", "Aug",
        "Sep", "Oct", "Nov", "Dec"
    ]
)
#Nummers omzetten in afkortingen van maanden.

fig.show()

### Revenue by channel

In [28]:
channel_sales = sales_df.merge(right=channel_df, on="ChannelKey", how="left", indicator=True)
#Aparte Dataframe maken waarbij Channel wordt gemerged met de salesdataset.
print(channel_sales.shape)
print(sales_df.shape)
#Shapes printen om te controleren of er evenveel records in de set zitten.

(2282482, 22)
(2282482, 20)


In [8]:
channel_revenue = (
    channel_sales
    .groupby("ChannelName", as_index=False)
    .agg(Revenue=("SalesAmountRet", "sum"))
    .sort_values("Revenue", ascending=True)
)

In [27]:
fig = px.bar(
    #Bar chart maken met behulp van channel_revenue dataframe.
    channel_revenue,
    x="Revenue",
    y="ChannelName",
    orientation="h",
    title="Revenue by sales channel"
)

fig.update_layout(
    xaxis_title="Revenue",
    yaxis_title="Sales channel",
    height=400,
    width = 800
)
#Titels voor axes instellen.
#Hoogte en wijdte van de chart instellen.

fig.update_xaxes(
    tickprefix="€",
    tickformat=",.0f"
)

fig.show()

### Revenue by product category

In [31]:
product_sales = (
    sales_df
    .merge(
        product_df,
        on="ProductKey",
        how="left",
        indicator=True
    )
    .merge(
        productcat_df,
        on="ProductSubcategoryKey", 
        how="left",
    )
)
print(f"product sales shape is {product_sales.shape}")
print(f"sales shape is {sales_df.shape}")

product sales shape is (2282482, 34)
sales shape is (2282482, 20)


In [37]:
category_revenue = (
    product_sales
    .groupby("ProductCategory", as_index=False)
    .agg(Revenue=("SalesAmountRet", "sum"))
    .sort_values("Revenue", ascending=True)
)

In [53]:
fig = px.pie(
    category_revenue,
    values="Revenue",
    names="ProductCategory",
    hole=0.8,
    title="Revenue by product category"
)

fig.update_layout(
    height=450,
    width=800
)

fig.update_traces(
    textposition="outside",
    textinfo="percent+label",
    showlegend=False
    
)


fig.show()

### GDP per capita versus revenue

In [58]:
GDP_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 795 entries, 0 to 794
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Country Name  795 non-null    str    
 1   Country Code  795 non-null    str    
 2   Year          795 non-null    int64  
 3   GDPPerCapita  779 non-null    float64
 4   CountryKey    795 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 52.9 KB


In [59]:
sales_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 20 columns):
 #   Column            Dtype         
---  ------            -----         
 0   SalesKey          int64         
 1   Date              datetime64[us]
 2   ChannelKey        int64         
 3   StoreKey          int64         
 4   ProductKey        int64         
 5   PromotionKey      int64         
 6   UnitCost          float64       
 7   UnitPrice         float64       
 8   SalesQuantity     int64         
 9   ReturnQuantity    int64         
 10  ReturnAmount      float64       
 11  DiscountQuantity  int64         
 12  DiscountAmount    float64       
 13  TotalCost         float64       
 14  SalesAmount       float64       
 15  SalesAmountRet    float64       
 16  TotalCostRet      float64       
 17  ProductUnitCost   float64       
 18  DateKey           int64         
 19  GeographyKey      int64         
dtypes: datetime64[us](1), float64(9), int64(10)
memory usage:

In [67]:
sales_geo = sales_df.merge(
    geography_df,
    on="GeographyKey",
    how="left"
)

sales_geo.info()
sales_geo["Year"] = sales_geo["Date"].dt.year

<class 'pandas.DataFrame'>
RangeIndex: 2282482 entries, 0 to 2282481
Data columns (total 23 columns):
 #   Column             Dtype         
---  ------             -----         
 0   SalesKey           int64         
 1   Date               datetime64[us]
 2   ChannelKey         int64         
 3   StoreKey           int64         
 4   ProductKey         int64         
 5   PromotionKey       int64         
 6   UnitCost           float64       
 7   UnitPrice          float64       
 8   SalesQuantity      int64         
 9   ReturnQuantity     int64         
 10  ReturnAmount       float64       
 11  DiscountQuantity   int64         
 12  DiscountAmount     float64       
 13  TotalCost          float64       
 14  SalesAmount        float64       
 15  SalesAmountRet     float64       
 16  TotalCostRet       float64       
 17  ProductUnitCost    float64       
 18  DateKey            int64         
 19  GeographyKey       int64         
 20  ContinentName      str           
 

In [ ]:
sales_gdp = sales_geo.merge(
    GDP_df,
    left_on=["RegionCountryName","Year"],
    right_on=["CountryKey","Year"],   # of jouw daadwerkelijke gemeenschappelijke kolom
    how="left"
)

print(len(sales_geo))
print(len(sales_gdp))
sales_gdp[["RegionCountryName", "CountryKey", "Year"]].head()

2282482
2282482


np.int64(7744)

In [77]:
country_year = (
    sales_gdp
    .groupby(["RegionCountryName", "Year"], as_index=False)
    .agg(
        Revenue=("SalesAmountRet", "sum"),
        GDP_per_capita=("GDPPerCapita", "mean")
    )
)

In [90]:
fig = px.scatter(
    country_year,
    x="GDP_per_capita",
    y="Revenue",
    color="Year",
    hover_name="RegionCountryName",
    title="Revenue vs GDP per capita",
    size="Revenue"
)

fig.update_layout(
    xaxis_title="GDP per capita",
    yaxis_title="Revenue",
    height=450,
    width=1000,
)

fig.update_yaxes(
    tickprefix="€",
    tickformat=",.0f"
)

fig.show()